# Nejoblíbenější dětská jména v ČR
**End-to-end pipeline:** discover → download → parse → visualise

Data source: [Český statistický úřad (ČSÚ)](https://csu.gov.cz) — press releases on baby-name popularity, covering years 2016–2019 (cumulative living-child counts) and 2022–2024 (annual births).

> **Note on methodology:** 2016–2019 files list *all* registered names for every living child in the Czech Republic as of January of that year (thousands of names ranked by count). The 2022–2024 files list births *in that calendar year* only (top 100 nationally, top 20 per region).

## 1. Dependencies

Install with `uv` (fast) or plain pip — only needed once.

In [1]:
# Uncomment to install
# import subprocess, sys
# subprocess.run([sys.executable, '-m', 'pip', 'install', 'requests', 'beautifulsoup4', 'openpyxl', 'pandas'], check=True)


In [2]:
import csv, json, re, os
from pathlib import Path
from collections import defaultdict
import requests
from bs4 import BeautifulSoup
import openpyxl
import pandas as pd
from IPython.display import display, HTML, IFrame


## 2. Source links

Seven press-release pages from ČSÚ — one per published year.

In [3]:
links_raw = Path('links.txt').read_text(encoding='utf-8')
print(links_raw)


https://csu.gov.cz/produkty/nejoblibenejsi-detska-jmena-jsou-jakub-a-eliska
https://csu.gov.cz/produkty/rodice-detem-nejcasteji-davaji-jmena-jan-a-eliska
https://csu.gov.cz/produkty/nejoblibenejsi-detska-jmena-jsou-jakub-a-eliska-fu0hnyje70
https://csu.gov.cz/produkty/eliska-a-jakub-opet-dominuji
https://csu.gov.cz/produkty/popularite-detskych-jmen-loni-vevodili-jakub-s-eliskou
https://csu.gov.cz/produkty/jmena-jakub-a-eliska-byla-loni-opet-nejoblibenejsi
https://csu.gov.cz/produkty/detskym-jmenum-loni-opet-kralovali-jakub-a-eliska



In [4]:
PAGES = [
    (2016, 'https://csu.gov.cz/produkty/nejoblibenejsi-detska-jmena-jsou-jakub-a-eliska'),
    (2017, 'https://csu.gov.cz/produkty/rodice-detem-nejcasteji-davaji-jmena-jan-a-eliska'),
    (2018, 'https://csu.gov.cz/produkty/nejoblibenejsi-detska-jmena-jsou-jakub-a-eliska-fu0hnyje70'),
    (2019, 'https://csu.gov.cz/produkty/eliska-a-jakub-opet-dominuji'),
    (2022, 'https://csu.gov.cz/produkty/popularite-detskych-jmen-loni-vevodili-jakub-s-eliskou'),
    (2023, 'https://csu.gov.cz/produkty/jmena-jakub-a-eliska-byla-loni-opet-nejoblibenejsi'),
    (2024, 'https://csu.gov.cz/produkty/detskym-jmenum-loni-opet-kralovali-jakub-a-eliska'),
]


## 3. Discover XLSX download links

Each page links to one or more `.xlsx` files. We scrape the href attributes.

In [5]:
BASE = 'https://csu.gov.cz'

def find_xlsx_links(url):
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, 'html.parser')
    found = []
    for a in soup.find_all('a', href=True):
        href = a['href']
        if '.xlsx' in href.lower():
            if not href.startswith('http'):
                href = BASE + href
            # strip query string for the filename, keep full URL for download
            fname = href.split('/')[-1].split('?')[0]
            found.append((fname, href))
    return found

discovered = {}
for year, url in PAGES:
    links = find_xlsx_links(url)
    discovered[year] = links
    print(f'{year}: {[f for f,_ in links]}')


2016: ['csu_detska_jmena_leden_2016_web.xlsx', 'csu_detska_jmena_leden_2016_web.xlsx']


2017: ['tabulka_s_celkovym_poradim.xlsx', 'tabulka_s_celkovym_poradim.xlsx']


2018: ['tabulka_s_celkovym_poradim.xlsx', 'tabulka_s_celkovym_poradim.xlsx']


2019: ['tabulka_s_celkovym_poradim_2019.xlsx', 'tabulka_s_celkovym_poradim_2019.xlsx']


2022: ['20_nejcetnejsich_detskych_jmen_kraje_2022.xlsx', '20_nejcetnejsich_detskych_jmen_kraje_2022.xlsx', '100_nejcetnejsich_detskych_jmen_republika_2022.xlsx', '100_nejcetnejsich_detskych_jmen_republika_2022.xlsx']


2023: ['20_nejcetnejsich_detskych_jmen_kraje_2023.xlsx', '20_nejcetnejsich_detskych_jmen_kraje_2023.xlsx', '100_nejcetnejsich_detskych_jmen_republika_2023.xlsx', '100_nejcetnejsich_detskych_jmen_republika_2023.xlsx']


2024: ['100_nejcetnejsich_detskych_jmen_republika_2024.xlsx', '100_nejcetnejsich_detskych_jmen_republika_2024.xlsx', '20_nejcetnejsich_detskych_jmen_kraje_2024.xlsx', '20_nejcetnejsich_detskych_jmen_kraje_2024.xlsx']


## 4. Download XLSX files

Files are saved to `raw/`. Already-downloaded files are skipped.

In [6]:
RAW = Path('raw')
RAW.mkdir(exist_ok=True)

# Map each (year, filename) to a local name that encodes year and type.
# Earlier years have a single 'overall' file; 2022–2024 have 'republika' and 'kraje'.
def local_name(year, fname):
    f = fname.lower()
    if 'kraje' in f:    return f'{year}_kraje.xlsx'
    if 'republika' in f: return f'{year}_republika.xlsx'
    return f'{year}_overall.xlsx'

to_download = []
for year, links in discovered.items():
    for fname, url in links:
        lname = local_name(year, fname)
        to_download.append((lname, url))

for lname, url in to_download:
    dest = RAW / lname
    if dest.exists():
        print(f'  skip  {lname}')
        continue
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    dest.write_bytes(r.content)
    print(f'  saved {lname}  ({len(r.content)//1024} KB)')

print('\nFiles in raw/:')
for p in sorted(RAW.iterdir()):
    print(f'  {p.name:40s}  {p.stat().st_size//1024:>4} KB')


  skip  2016_overall.xlsx
  skip  2016_overall.xlsx
  skip  2017_overall.xlsx
  skip  2017_overall.xlsx
  skip  2018_overall.xlsx
  skip  2018_overall.xlsx
  skip  2019_overall.xlsx
  skip  2019_overall.xlsx
  skip  2022_kraje.xlsx
  skip  2022_kraje.xlsx
  skip  2022_republika.xlsx
  skip  2022_republika.xlsx
  skip  2023_kraje.xlsx
  skip  2023_kraje.xlsx
  skip  2023_republika.xlsx
  skip  2023_republika.xlsx
  skip  2024_republika.xlsx
  skip  2024_republika.xlsx
  skip  2024_kraje.xlsx
  skip  2024_kraje.xlsx

Files in raw/:
  2016_overall.xlsx                           30 KB
  2017_overall.xlsx                           27 KB
  2018_overall.xlsx                           30 KB
  2019_overall.xlsx                           32 KB
  2022_kraje.xlsx                            536 KB
  2022_republika.xlsx                         13 KB
  2023_kraje.xlsx                            536 KB
  2023_republika.xlsx                         14 KB
  2024_kraje.xlsx                            536

## 5. Inspect file structures

The files fall into three distinct formats — we need to understand each before writing the parser.

### 5a. 2016–2019 — single file, Czech Republic only

One sheet per file, boys and girls listed side-by-side in paired columns.

In [7]:
for year in [2016, 2017, 2018, 2019]:
    wb = openpyxl.load_workbook(RAW / f'{year}_overall.xlsx', data_only=True)
    ws = wb.active
    rows = [r for r in ws.iter_rows(values_only=True) if any(c is not None for c in r)]
    print(f'--- {year}  sheets={wb.sheetnames}  rows={ws.max_row}  cols={ws.max_column}')
    print(f'    header: {rows[0]}')
    for r in rows[1:4]: print(f'    {r}')
    print()


--- 2016  sheets=['Leden 2016']  rows=620  cols=6
    header: (None, 'Pořadí', 'Kluci jména', None, 'Pořadí', 'Holky jména')
    (None, 1, 'Jakub', None, 1, 'Eliška')
    (None, 2, 'Jan', None, 2, 'Tereza')
    (None, 3, 'Tomáš', None, 3, 'Anna')

--- 2017  sheets=['List1']  rows=594  cols=7
    header: (None, None, 'Pořadí', 'Kluci jména', None, 'Pořadí', 'Holky jména')
    (None, None, 1, 'Jan', None, 1, 'Eliška')
    (None, None, 2, 'Jakub', None, 2, 'Tereza')
    (None, None, 3, 'Matyáš', None, 3, 'Anna')

--- 2018  sheets=['List1']  rows=681  cols=7
    header: (None, None, 'Pořadí', 'Kluci jména', None, 'Pořadí', 'Holky jména')
    (None, None, 1, 'Jakub', None, 1, 'Eliška')
    (None, None, 2, 'Jan', None, 2, 'Anna')
    (None, None, 3, 'Adam', None, 3, 'Sofie')

--- 2019  sheets=['List1']  rows=681  cols=7
    header: ('Leden 2019', None, 'Pořadí', 'Kluci jména', None, 'Pořadí', 'Holky jména')
    (None, None, 1, 'Jakub', None, 1, 'Eliška')
    (None, None, 2, 'Jan', None, 2, '

### 5b. 2022–2024 republika — two sheets (Chlapci / Dívky), top 100

Simple two-column layout: rank + name.

In [8]:
for year in [2022, 2023, 2024]:
    wb = openpyxl.load_workbook(RAW / f'{year}_republika.xlsx', data_only=True)
    for sn in wb.sheetnames:
        ws = wb[sn]
        rows = list(ws.iter_rows(min_row=1, max_row=4, values_only=True))
        print(f'{year} [{sn}]  {ws.max_row-1} data rows')
        for r in rows: print(f'    {r}')
    print()


2022 [Chlapci]  100 data rows
    ('Pořadí v ČR', 'Jméno')
    (1, 'JAKUB')
    (2, 'JAN')
    (3, 'MATYÁŠ')
2022 [Dívky]  100 data rows
    ('Pořadí v ČR', 'Jméno')
    (1, 'ELIŠKA')
    (2, 'VIKTORIE')
    (3, 'ANNA')

2023 [Chlapci]  105 data rows
    ('Pořadí v ČR', 'Jméno')
    (1, 'JAKUB')
    (2, 'MATYÁŠ')
    (3, 'JAN')
2023 [Dívky]  107 data rows
    ('Pořadí v ČR', 'Jméno')
    (1, 'ELIŠKA')
    (2, 'VIKTORIE')
    (3, 'ANNA')

2024 [Chlapci]  105 data rows
    ('Pořadí v ČR', 'Jméno')
    (1, 'JAKUB')
    (2, 'MATYÁŠ')
    (3, 'JAN')
2024 [Dívky]  107 data rows
    ('Pořadí v ČR', 'Jméno')
    (1, 'ELIŠKA')
    (2, 'VIKTORIE')
    (3, 'SOFIE')



### 5c. 2022–2024 kraje — two sheets, 15 regions × 3 columns (rank, name, spacer)

Row 1 = region names, Row 2 = column headers, Rows 3+ = data (top ~20 per region).

In [9]:
wb = openpyxl.load_workbook(RAW / '2022_kraje.xlsx', data_only=True)
ws = wb['Chlapci']
rows = list(ws.iter_rows(values_only=True))
print('Header row (region names):')
print(' | '.join(str(v) for v in rows[0] if v is not None))
print(f'\nData rows per region (Chlapci):')
for i, region in enumerate(['Česko','Praha','Středočeský','Jihočeský','Plzeňský',
                              'Karlovarský','Ústecký','Liberecký','Královéhradecký',
                              'Pardubický','Vysočina','Jihomoravský','Olomoucký',
                              'Zlínský','Moravskoslezský']):
    col_n = i * 3 + 1
    count = sum(1 for r in rows[2:] if col_n < len(r) and r[col_n] is not None)
    print(f'  {region:<22} {count} names')


Header row (region names):
Česko | Praha | Středočeský | Jihočeský | Plzeňský | Karlovarský | Ústecký | Liberecký | Královéhradecký | Pardubický | Vysočina | Jihomoravský | Olomoucký | Zlínský | Moravskoslezský

Data rows per region (Chlapci):
  Česko                  20 names
  Praha                  20 names
  Středočeský            21 names
  Jihočeský              22 names
  Plzeňský               20 names
  Karlovarský            20 names
  Ústecký                20 names
  Liberecký              20 names
  Královéhradecký        20 names
  Pardubický             21 names
  Vysočina               21 names
  Jihomoravský           20 names
  Olomoucký              20 names
  Zlínský                21 names
  Moravskoslezský        22 names


## 6. Parse into unified CSV

All formats are normalised to a single schema: `year, region, gender, rank, name`.

**Strategy:**
- 2016–2019: detect column offsets from the header row; extract boys/girls pairs; region = `Česko`
- 2022–2024 kraje: primary source — includes all 15 regions, top ~20 names each
- 2022–2024 republika: supplement — adds ranks 21–100 for `Česko` that kraje doesn't cover

In [10]:
REGIONS_KRAJE = [
    'Česko', 'Praha', 'Středočeský', 'Jihočeský', 'Plzeňský',
    'Karlovarský', 'Ústecký', 'Liberecký', 'Královéhradecký', 'Pardubický',
    'Vysočina', 'Jihomoravský', 'Olomoucký', 'Zlínský', 'Moravskoslezský',
]

def parse_rank(val):
    if val is None: return None
    return str(val).strip().replace('–', '-').replace('—', '-')

records = []

def parse_old_format(path, year):
    wb = openpyxl.load_workbook(path, data_only=True)
    ws = wb.active
    rows = list(ws.iter_rows(values_only=True))
    header = rows[0]
    boy_rank_col = next(i for i, v in enumerate(header) if v == 'Pořadí')
    boy_name_col = boy_rank_col + 1
    girl_rank_col = next(i for i, v in enumerate(header) if v == 'Pořadí' and i > boy_rank_col)
    girl_name_col = girl_rank_col + 1
    for row in rows[1:]:
        b_rank, b_name = row[boy_rank_col], row[boy_name_col]
        g_rank, g_name = row[girl_rank_col], row[girl_name_col]
        if b_name: records.append(dict(year=year, region='Česko', gender='boy',  rank=parse_rank(b_rank), name=str(b_name).strip()))
        if g_name: records.append(dict(year=year, region='Česko', gender='girl', rank=parse_rank(g_rank), name=str(g_name).strip()))

def parse_republika(path, year):
    wb = openpyxl.load_workbook(path, data_only=True)
    for sn, gender in [('Chlapci', 'boy'), ('Dívky', 'girl')]:
        for row in list(wb[sn].iter_rows(values_only=True))[1:]:
            rank, name = row[0], row[1]
            if name: records.append(dict(year=year, region='Česko', gender=gender, rank=parse_rank(rank), name=str(name).strip().title()))

def parse_kraje(path, year):
    wb = openpyxl.load_workbook(path, data_only=True)
    for sn, gender in [('Chlapci', 'boy'), ('Dívky', 'girl')]:
        rows = list(wb[sn].iter_rows(values_only=True))
        for ri, region in enumerate(REGIONS_KRAJE):
            rank_col, name_col = ri * 3, ri * 3 + 1
            for row in rows[2:]:
                rank = row[rank_col] if rank_col < len(row) else None
                name = row[name_col] if name_col < len(row) else None
                if name: records.append(dict(year=year, region=region, gender=gender, rank=parse_rank(rank), name=str(name).strip().title()))

# --- run all parsers ---
for year in [2016, 2017, 2018, 2019]:
    parse_old_format(RAW / f'{year}_overall.xlsx', year)

for year in [2022, 2023, 2024]:
    parse_kraje(RAW / f'{year}_kraje.xlsx', year)
    # add republika ranks beyond what kraje covers (~21-100)
    wb_rep = openpyxl.load_workbook(RAW / f'{year}_republika.xlsx', data_only=True)
    kraje_max = {}
    for r in records:
        if r['year'] == year and r['region'] == 'Česko':
            try:
                rk = int(r['rank'].split('-')[0])
                kraje_max[r['gender']] = max(kraje_max.get(r['gender'], 0), rk)
            except (ValueError, AttributeError): pass
    for sn, gender in [('Chlapci', 'boy'), ('Dívky', 'girl')]:
        threshold = kraje_max.get(gender, 0)
        for row in list(wb_rep[sn].iter_rows(values_only=True))[1:]:
            rank, name = row[0], row[1]
            if name and isinstance(rank, int) and rank > threshold:
                records.append(dict(year=year, region='Česko', gender=gender, rank=parse_rank(rank), name=str(name).strip().title()))

# write CSV
OUT = Path('names.csv')
with open(OUT, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['year','region','gender','rank','name'])
    writer.writeheader()
    writer.writerows(records)

print(f'Wrote {len(records):,} records → {OUT}')


Wrote 6,906 records → names.csv


## 7. Data preview

Quick look at the unified dataset.

In [11]:
df = pd.read_csv('names.csv')
print(df.shape)
df.head(10)


(6906, 5)


,year,region,gender,rank,name
0,2016,Česko,boy,1,Jakub
1,2016,Česko,girl,1,Eliška
2,2016,Česko,boy,2,Jan
3,2016,Česko,girl,2,Tereza
4,2016,Česko,boy,3,Tomáš
5,2016,Česko,girl,3,Anna
6,2016,Česko,boy,4,Filip
7,2016,Česko,girl,4,Adéla
8,2016,Česko,boy,5,Ondřej
9,2016,Česko,girl,5,Sofie


In [12]:
print('Years covered:', sorted(df.year.unique()))
print('Regions:', sorted(df.region.unique()))
print()
print('Records per year × gender (Česko only):')
print(df[df.region=='Česko'].groupby(['year','gender']).size().unstack())


Years covered: [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2022), np.int64(2023), np.int64(2024)]
Regions: ['Jihomoravský', 'Jihočeský', 'Karlovarský', 'Královéhradecký', 'Liberecký', 'Moravskoslezský', 'Olomoucký', 'Pardubický', 'Plzeňský', 'Praha', 'Středočeský', 'Vysočina', 'Zlínský', 'Ústecký', 'Česko']

Records per year × gender (Česko only):
gender  boy  girl
year             
2016    517   619
2017    496   593
2018    555   680
2019    582   678
2022     86    80
2023     90    77
2024     71    67


In [13]:
print('Top 5 boys by year (Česko):')
top5 = (
    df[(df.region=='Česko') & (df.gender=='boy')]
    .assign(rank_int=lambda d: pd.to_numeric(d['rank'].str.split('-').str[0], errors='coerce'))
    .query('rank_int <= 5')
    .sort_values(['year','rank_int'])
    .groupby('year')['name'].apply(list)
)
for yr, names in top5.items():
    print(f'  {yr}: {names}')


Top 5 boys by year (Česko):
  2016: ['Jakub', 'Jan', 'Tomáš', 'Filip', 'Ondřej']
  2017: ['Jan', 'Jakub', 'Matyáš', 'Filip', 'Lukáš']
  2018: ['Jakub', 'Jan', 'Adam', 'Matyáš', 'Tomáš']
  2019: ['Jakub', 'Jan', 'Tomáš', 'Matyáš', 'Adam']
  2022: ['Jakub', 'Jan', 'Matyáš', 'Adam', 'Tomáš']
  2023: ['Jakub', 'Matyáš', 'Jan', 'Adam', 'Matěj']
  2024: ['Jakub', 'Matyáš', 'Jan', 'Adam', 'Matěj']


## 8. Generate the interactive HTML visualisation

A self-contained bump chart in plain HTML + vanilla JS:
- sticky single-line toolbar (gender, region, top-N, search)
- Bézier connectors between years, dashed across the 2020–21 gap
- hover → highlight one name's trajectory + show rank
- search (≥ 3 chars) → highlight matching names with rank labels

In [14]:
# ── data → JSON ──────────────────────────────────────────────────────────
YEARS_VIZ = [2016, 2017, 2018, 2019, 2022, 2023, 2024]

def read_data_for_viz():
    data = {}
    with open('names.csv', encoding='utf-8') as f:
        for r in csv.DictReader(f):
            y, reg, g = int(r['year']), r['region'], r['gender']
            try:
                rank = int(r['rank'].split('-')[0])
            except (ValueError, AttributeError):
                rank = 9999
            data.setdefault(y, {}).setdefault(reg, {'boy': [], 'girl': []})
            data[y][reg][g].append({'rank': rank, 'name': r['name']})
    for y in data:
        for reg in data[y]:
            for g in ('boy', 'girl'):
                data[y][reg][g].sort(key=lambda x: x['rank'])
    return data

data_viz = read_data_for_viz()
all_regions = sorted({r for y in data_viz for r in data_viz[y]})
regions = ['Česko'] + [r for r in all_regions if r != 'Česko']

out = {}
for y in YEARS_VIZ:
    out[y] = {}
    for reg in regions:
        e = data_viz.get(y, {}).get(reg, {'boy': [], 'girl': []})
        out[y][reg] = {'boy': e['boy'][:100], 'girl': e['girl'][:100]}

data_json   = json.dumps(out, ensure_ascii=False, separators=(',', ':'))
region_opts = '\n'.join(f'      <option value="{r}">{r}</option>' for r in regions)


In [15]:
# ── HTML template (truncated for display — full version in generate_html.py) ──
# Read the template from generate_html.py and substitute placeholders
import importlib.util, sys

spec = importlib.util.spec_from_file_location('gen', 'generate_html.py')
gen_mod = importlib.util.load_from_spec = None  # avoid running main

# Simpler: just read the file and extract HTML_TEMPLATE
src = Path('generate_html.py').read_text(encoding='utf-8')
# The template is everything between HTML_TEMPLATE = r""" and the final """
match = re.search(r'HTML_TEMPLATE = r"""(.+?)"""', src, re.DOTALL)
HTML_TEMPLATE = match.group(1)

html = HTML_TEMPLATE.replace('%%DATA%%', data_json).replace('%%REGIONS%%', region_opts)
Path('index.html').write_text(html, encoding='utf-8')
print(f'Generated index.html  ({len(html)//1024} KB)')


Generated index.html  (91 KB)


## 9. Visualisation

The chart is embedded below via `<iframe>`. If it appears blank, open the link directly — some Jupyter environments restrict inline frames.

In [16]:
link = Path('index.html').resolve().as_uri()
display(HTML(
    f'<p style="margin-bottom:8px">'
    f'<a href="{link}" target="_blank" '
    f'style="font-size:14px;font-weight:600;color:#2563eb;text-decoration:none;'
    f'padding:6px 12px;border:1px solid #2563eb;border-radius:6px;">'
    f'↗ Open index.html in new tab</a></p>'
))

# Embed directly — works in JupyterLab / classic Notebook when served from the same dir
display(IFrame(src='index.html', width='100%', height=720))


---

*Data: Český statistický úřad (ČSÚ) · Notebook generated automatically · Visualisation: plain HTML + vanilla JS, no external dependencies.*